# Position matching EDA

Protein-level context for ClinVar variants joined to UniProt.

Built by `python src/position_matching_clinvar_uniprot.py` → `data/processed/clinvar_uniprot_position_matched.parquet` (from the gene-level `clinvar_uniprot_joined.parquet`). Adds:

- `protein_position` — amino-acid index from ClinVar HGVS `p.` notation
- boolean overlap flags against UniProt feature intervals (domain, region, active site, binding site, zinc finger, disulfide, modified residue)
- `closest_feature_type` / `closest_feature_description` — nearest UniProt annotated feature (ties break toward functional sites)
- `distance_to_closest_feature` — AA distance to that feature (`0` if the variant sits on it)

Gene-level join is unchanged; this step adds **where on the protein** each variant falls.

In [21]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv

In [22]:
load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
POSITION_MATCHED_PARQUET = project_root / "data/processed/clinvar_uniprot_position_matched.parquet"

In [23]:
FEATURE_FLAGS = [
    "in_domain",
    "in_region",
    "in_zinc_finger",
    "in_active_site",
    "in_binding_site",
    "in_disulfide",
    "in_mod_res",
    "in_functional_site",
    "in_any_feature",
]


def pct(n, d):
    return round(100 * n / d, 1) if d else 0.0

In [24]:
df = pd.read_parquet(POSITION_MATCHED_PARQUET)
both = df[df["match_type"] == "both"].copy()
both_with_pos = both[both["has_protein_position"]].copy()

print("rows:", df.shape)
print("matched (both):", len(both))
print("with protein position:", len(both_with_pos))

rows: (16863, 55)
matched (both): 11586
with protein position: 9024


## Headline results (`match_type == both`)

In [ ]:
n_matched = len(both)
with_pos = int(both["has_protein_position"].sum())
in_domain = int(both["in_domain"].sum())
in_functional = int(both["in_functional_site"].sum())
has_closest = int(both_with_pos["closest_feature_type"].notna().sum())
median_dist = both_with_pos["distance_to_closest_feature"].median()

print(f"{with_pos:,} / {n_matched:,} ({with_pos / n_matched:.1%}) have a protein position")
print(f"{in_domain:,} ({in_domain / n_matched:.1%}) fall in a UniProt domain")
print(
    f"{in_functional:,} ({in_functional / n_matched:.1%}) "
    "fall in active/binding/zinc-finger sites"
)
print(
    f"{has_closest:,} / {with_pos:,} ({has_closest / with_pos:.1%}) "
    "have a closest UniProt feature"
)
print(f"median distance to closest feature: {median_dist:.0f} AA")


## Summary table — coverage and feature overlap

In [26]:
summary_rows = [
    {
        "metric": "Matched rows (gene join)",
        "count": len(both),
        "pct_of_matched": 100.0,
        "pct_of_with_pos": float("nan"),  # not a subset of with-pos
    },
    {
        "metric": "Has protein position (p. notation)",
        "count": int(both["has_protein_position"].sum()),
        "pct_of_matched": pct(both["has_protein_position"].sum(), len(both)),
        "pct_of_with_pos": 100.0,
    },
]

for flag in FEATURE_FLAGS:
    n_all = int(both[flag].sum())
    n_pos = int(both_with_pos[flag].sum())
    summary_rows.append(
        {
            "metric": flag,
            "count": n_all,
            "pct_of_matched": pct(n_all, len(both)),
            "pct_of_with_pos": pct(n_pos, len(both_with_pos)),
        }
    )

summary = pd.DataFrame(summary_rows).set_index("metric")
summary

,count,pct_of_matched,pct_of_with_pos
metric,,,
Matched rows (gene join),11586,100.0,NaN
Has protein position (p. notation),9024,77.9,100.0
in_domain,2360,20.4,26.2
in_region,2686,23.2,29.8
in_zinc_finger,64,0.6,0.7
in_active_site,5,0.0,0.1
in_binding_site,139,1.2,1.5
in_disulfide,647,5.6,7.2
in_mod_res,68,0.6,0.8


Intronic/splice variants lack `p.` notation, so `pct_of_with_pos` is the meaningful denominator for feature overlap. Low active-site coverage is expected — only enzyme-type proteins carry that annotation.

## Summary table — unique ClinVar variants (deduplicated by VariationID)

In [27]:
def unique_variant_counts(frame: pd.DataFrame) -> pd.Series:
    grouped = frame.groupby("VariationID")
    return pd.Series(
        {
            "variants": grouped.ngroups,
            "has_protein_position": grouped["has_protein_position"].any().sum(),
            **{flag: grouped[flag].any().sum() for flag in FEATURE_FLAGS},
        }
    )


variant_summary = pd.DataFrame(
    {
        "all_matched": unique_variant_counts(both),
        "pathogenic": unique_variant_counts(both[both["label"] == "pathogenic"]),
        "benign": unique_variant_counts(both[both["label"] == "benign"]),
    }
).astype(int)

variant_summary.loc["pct_has_protein_position"] = (
    100 * variant_summary.loc["has_protein_position"] / variant_summary.loc["variants"]
).round(1)
variant_summary.loc["pct_in_domain"] = (
    100 * variant_summary.loc["in_domain"] / variant_summary.loc["has_protein_position"]
).round(1)
variant_summary.loc["pct_in_functional_site"] = (
    100 * variant_summary.loc["in_functional_site"] / variant_summary.loc["has_protein_position"]
).round(1)

variant_summary

,all_matched,pathogenic,benign
variants,11563.0,6352.0,5211.0
has_protein_position,9018.0,5376.0,3642.0
in_domain,2358.0,1760.0,598.0
in_region,2686.0,1272.0,1414.0
in_zinc_finger,64.0,55.0,9.0
in_active_site,5.0,5.0,0.0
in_binding_site,139.0,125.0,14.0
in_disulfide,647.0,512.0,135.0
in_mod_res,68.0,40.0,28.0
in_functional_site,205.0,182.0,23.0


## Summary table — pathogenic vs benign by feature overlap (variants with protein position)

In [28]:
label_feature_rows = []
for label in ["pathogenic", "benign"]:
    subset = both_with_pos[both_with_pos["label"] == label]
    row = {"label": label, "variants_with_pos": len(subset)}
    for flag in FEATURE_FLAGS:
        n = int(subset[flag].sum())
        row[flag] = n
        row[f"{flag}_pct"] = pct(n, len(subset))
    label_feature_rows.append(row)

label_feature = pd.DataFrame(label_feature_rows).set_index("label")
label_feature[[c for c in label_feature.columns if not c.endswith("_pct")]]

,variants_with_pos,in_domain,in_region,in_zinc_finger,in_active_site,in_binding_site,in_disulfide,in_mod_res,in_functional_site,in_any_feature
label,,,,,,,,,,
pathogenic,5381,1761,1272,55,5,125,512,40,182,2902
benign,3643,599,1414,9,0,14,135,28,23,2023


### Biological intuition

Expected feature importance for the pathognicity classifier, ordered higest-lowest:

1. 
`in_active_site` - most conserved, mutations disrupt catalysis,  likely to lead to loss of function. 

`in_zinc_finger` - variants in histidine / cysteine residues completely distabilising the DNA binding. 

2.
`in_disulfide` - mutating either of the 2 cysteine residues that the disulfie bond connects can distabilize the fold.

`in_binding_site` - involved in interactions with ligands, substrates, metal ions, cofactors. Sometimes binding affinity can be partially perserved even after mutation.

3. 
`is_domain` - baseline signal, important functional region but it can contain scaffolding around the actual chemistry sites, where variants are better tolerated

4. 
`in_region` - conditional importance, some of them can tolerate mutations, e.g. intristically disordered regions

`in_mod_res` - conditional importance, sometimes mutation in modified residues can impact signaling without destroying protein fold and function

## Top domains hit by pathogenic variants

In [29]:
path_in_domain = both_with_pos[
    (both_with_pos["label"] == "pathogenic") & both_with_pos["in_domain"]
]

domain_counts = (
    path_in_domain["domain_names"]
    .dropna()
    .str.split("; ")
    .explode()
    .value_counts()
    .head(15)
    .rename("pathogenic_variants")
    .to_frame()
)
domain_counts

,pathogenic_variants
domain_names,
Hexokinase,279
Protein kinase,79
Phosphatase tensin-type,76
calcium-binding,64
POU-specific atypical,53
ACT,51
Olfactomedin-like,47
BRCT 2,46
RMT2,46


## Closest UniProt feature + distance

Binary `in_*` flags are sparse (~55% of coding variants sit outside every annotated interval). For each variant with a protein position we also record:

- `closest_feature_type` — UniProt FT type of the nearest annotation
- `closest_feature_description` — `/note` text when present
- `distance_to_closest_feature` — AA residues to that feature (`0` = on it)

Disulfide bonds use the bonded cysteine endpoints (not the intervening span). On distance ties, functional sites win over domains/regions.

In [ ]:
with_closest = both_with_pos[both_with_pos["closest_feature_type"].notna()].copy()

print(
    f"{len(with_closest):,} / {len(both_with_pos):,} "
    f"({len(with_closest) / len(both_with_pos):.1%}) have a closest UniProt feature"
)
print(
    f"{(with_closest['distance_to_closest_feature'] == 0).sum():,} "
    f"({(with_closest['distance_to_closest_feature'] == 0).mean():.1%}) sit on the closest feature"
)
print(
    f"distance — median {with_closest['distance_to_closest_feature'].median():.0f}, "
    f"mean {with_closest['distance_to_closest_feature'].mean():.1f}"
)

type_counts = (
    with_closest["closest_feature_type"]
    .value_counts()
    .rename("variants")
    .to_frame()
)
type_counts["pct"] = (100 * type_counts["variants"] / len(with_closest)).round(1)
type_counts

In [ ]:
dist_by_label = (
    with_closest.groupby("label")["distance_to_closest_feature"]
    .agg(n="count", median="median", mean="mean", p75=lambda s: s.quantile(0.75))
    .round(1)
)
display(dist_by_label)

outside = with_closest[with_closest["distance_to_closest_feature"] > 0]
outside_by_label = (
    outside.groupby("label")["distance_to_closest_feature"]
    .agg(n="count", median="median", mean="mean")
    .round(1)
)
print("Among variants with distance > 0 (outside every annotated interval):")
outside_by_label


## Spot check — BRCA1

In [ ]:
brca_cols = [
    "Name",
    "protein_position",
    "label",
    "closest_feature_type",
    "closest_feature_description",
    "distance_to_closest_feature",
    "in_domain",
    "domain_names",
    "in_region",
    "in_zinc_finger",
]
both[both["gene"] == "BRCA1"][brca_cols].head(10)


e.g. benign variant but in_region=True (NM_007294.4(BRCA1):c.3607C>T (p.Arg1203Ter))

Most likely it falls in one of BRCA1's large unstructured linker regions — UniProt annotates these as "Region" precisely because they aren't a functional domain. The naming is confusing: REGION in UniProt doesn't mean "functionally important region" — it means "a stretch of sequence with some noted property," which could be:
- "Interaction with X"  ← functionally important
- "Disordered"          ← structurally unimportant
- "Required for X"      ← important
- "Low complexity"      ← probably not critical
- "Interaction with BARD1, coiled-coil" ← important

So in_region = True is a weak and heterogeneous signal. You need to go one level deeper — what the region's description actually says — rather than treating all regions as equivalent.
Arg1443Gly specifically: position 1443 is in BRCA1's large central domain (roughly aa 300-1600) which contains many interaction motifs but is also substantially tolerant of variation — the gnomAD allele frequency and evolutionary conservation at that specific position will tell you more than domain membership

BRCA1 has 1863 amino acids
Formally annotated positions:
  - RING domain:     aa 1-109       (109 residues, 6%)
  - Zinc finger:     aa 27-65       (within RING)
  - BRCT domains:    aa 1646-1863   (217 residues, 12%)
  - Unannotated:       aa 110-1645    (1535 residues, 82%)

And BRCA1 is one of the better annotated proteins. For less studied genes the unannotated fraction is much higher.
This doesn't mean the join is worthless — it means UniProt domain features are sparse but high precision when present. A variant in the BRCA1 RING domain zinc finger is almost certainly meaningful. But the feature is zero for most variants.